In [1]:
# --- imports ---
import pandas as pd
import sqlite3

print("All imports OK")

All imports OK


In [2]:
# --- config ---
MODE = "bus"
DATASET = "../data/ttc-bus-delay-data-2025.csv"
LOOKUP = "../data/ttc-bus-delay-data-lookup.csv"
DB = "../sql/TTC_delay.db"

In [3]:
df = pd.read_csv(DATASET)
df.head()

,_id,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle
0,1,2025-01-01T00:00:00,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442
1,2,2025-01-01T00:00:00,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,NaN,0
2,3,2025-01-01T00:00:00,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,NaN,8546
3,4,2025-01-01T00:00:00,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693
4,5,2025-01-01T00:00:00,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102257 entries, 0 to 102256
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   _id        102257 non-null  int64
 1   Date       102257 non-null  str  
 2   Line       101016 non-null  str  
 3   Time       102257 non-null  str  
 4   Day        102257 non-null  str  
 5   Station    102257 non-null  str  
 6   Code       102257 non-null  str  
 7   Min Delay  102257 non-null  int64
 8   Min Gap    102257 non-null  int64
 9   Bound      80930 non-null   str  
 10  Vehicle    102257 non-null  int64
dtypes: int64(4), str(7)
memory usage: 8.6 MB


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isna().sum()

_id              0
Date             0
Line          1241
Time             0
Day              0
Station          0
Code             0
Min Delay        0
Min Gap          0
Bound        21327
Vehicle          0
dtype: int64

In [7]:
# catching stray values
df["Bound"].value_counts(dropna=False)

Bound
N      22742
S      21881
NaN    21327
E      18729
W      17522
B         33
M          6
D          4
8          2
4          2
A          2
2          1
7          1
G          1
`          1
Z          1
Q          1
3          1
Name: count, dtype: int64

In [8]:
valid_directions = ['N', 'S', 'E', 'W']

#  group anything that is NOT a valid direction (including NaNs) and set it to "Unknown"
df.loc[~df['Bound'].isin(valid_directions), 'Bound'] = 'Unknown'

# recheck
df["Bound"].value_counts(dropna=False)

Bound
N          22742
S          21881
Unknown    21383
E          18729
W          17522
Name: count, dtype: int64

In [9]:
# standardize column names
df = df.rename(columns={"_id": "source_id",
                        "Min Delay": "min_delay", "Min Gap": "min_gap",})
df.columns = df.columns.str.lower()

# keep missing bus line
df["line"] = df["line"].fillna("Unknown") 

# time format
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
df["quarter"]  = df["date"].dt.year.astype(str) + "-Q" + df["date"].dt.quarter.astype(str)
df["hour"]  = df["time"].str[:2].astype(int)

# new identifiers
df["is_delay"] = (df["min_delay"] > 0).astype(int)
df["source_id"] = MODE + "-" + df["source_id"].astype(str)


df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102257 entries, 0 to 102256
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   source_id  102257 non-null  str           
 1   date       102257 non-null  datetime64[us]
 2   line       102257 non-null  str           
 3   time       102257 non-null  str           
 4   day        102257 non-null  str           
 5   station    102257 non-null  str           
 6   code       102257 non-null  str           
 7   min_delay  102257 non-null  int64         
 8   min_gap    102257 non-null  int64         
 9   bound      102257 non-null  str           
 10  vehicle    102257 non-null  int64         
 11  month      102257 non-null  str           
 12  quarter    102257 non-null  str           
 13  hour       102257 non-null  int64         
 14  is_delay   102257 non-null  int64         
dtypes: datetime64[us](1), int64(5), str(9)
memory usage: 11.7 MB


In [10]:
# load lookup table and do basic cleaning
lookup = pd.read_csv(LOOKUP, encoding="cp1252")
lookup.columns = lookup.columns.str.lower()
lookup = lookup[["code", "description"]].drop_duplicates("code")

# join table on "code"
df = df.merge(lookup, on="code", how="left")
df["description"] = df["description"].fillna("Unknown")

df.head()

,source_id,date,line,time,day,station,code,min_delay,min_gap,bound,vehicle,month,quarter,hour,is_delay,description
0,bus-1,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442,2025-01,2025-Q1,2,1,NO OPERATOR AVAILABLE DUE TO ESA
1,bus-2,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,Unknown,0,2025-01,2025-Q1,2,0,UNABLE TO MAINTAIN SCHEDULE
2,bus-3,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,Unknown,8546,2025-01,2025-Q1,2,0,ILL CUSTOMER - TRANSPORTED
3,bus-4,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693,2025-01,2025-Q1,2,1,UNSANITARY
4,bus-5,2025-01-01,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801,2025-01,2025-Q1,3,1,ILL CUSTOMER - TRANSPORTED


In [11]:
df.tail()

,source_id,date,line,time,day,station,code,min_delay,min_gap,bound,vehicle,month,quarter,hour,is_delay,description
102252,bus-102253,2026-08-31,36 FINCH WEST,00:41,Monday,FINCH WEST STATION,EFCEL,5,15,E,9149,2026-08,2026-Q3,0,1,Unknown
102253,bus-102254,2026-08-31,63 OSSINGTON,00:50,Monday,CEDARVALE,MFSH,8,16,N,8384,2026-08,2026-Q3,0,1,USED AS SHUTTLE BUS
102254,bus-102255,2026-08-31,72 PAPE,00:57,Monday,DON VALLEY STATION,MFSH,14,30,W,8918,2026-08,2026-Q3,0,1,USED AS SHUTTLE BUS
102255,bus-102256,2026-08-31,35 JANE,01:14,Monday,STEELES AND MURRAY ROS,MFUS,20,40,S,7273,2026-08,2026-Q3,1,1,UNABLE TO MAINTAIN SCHEDULE
102256,bus-102257,2026-08-31,106 SENTINEL,01:18,Monday,SETINEL AND GRANDRAVIN,MFSH,30,60,E,7006,2026-08,2026-Q3,1,1,USED AS SHUTTLE BUS


In [12]:
# sqlite
conn = sqlite3.connect(DB)
df.to_sql(f"{MODE}_delays", conn, if_exists="replace", index=False)

groups = {"month": "month", "hour": "hour", "reason": "description"}
for name, col in groups.items():
    conn.execute(f"DROP VIEW IF EXISTS view_{MODE}_by_{name}")
    conn.execute(f"""
    CREATE VIEW view_{MODE}_by_{name} AS
    SELECT {col},
           COUNT(*) AS incidents,
           SUM(min_delay) AS total_delay_min,
           ROUND(AVG(CASE WHEN is_delay = 1 THEN min_delay END), 2) AS avg_delay_min
    FROM {MODE}_delays
    GROUP BY {col}""")
conn.commit()

print(f"{MODE} view created successfully.")
conn.close()

bus view created successfully.
